# מפגש 25 — רגרסיה משולבת ו־MAE | מעבדה לתלמיד
עובדים לפי המשימות המקוריות: `bill_depth_mm`, `flipper_length_mm`, `island` לניבוי `body_mass_g`, חלוקה seed=17, חמשת קיפולי KFold באימון בלבד. **השלימו ???.** הנתונים בתרגול החישוב הראשוני מומצאים; אין כאן תוצאה מוכנה על Penguins.

## שלב 0 — חישוב ידני: למה לא ממוצע שאריות חתומות?
אמת [100,200,300] וחיזוי [110,180,330]. טעויות מוחלטות [10,20,30], ולכן MAE = (10+20+30)/3. כתבו את משמעות יחידת המידה.

In [ ]:
import numpy as np
toy_actual=np.array([100,200,300])
toy_pred=np.array([110,180,330])
toy_mae=np.mean(np.???(toy_actual-toy_pred))
assert np.isclose(toy_mae,???)
print("Toy MAE:",toy_mae)

## שלב 1 — טעינה, יעד ופיצול
היעד הוא מסה בגרמים; X כולל 2 מדידות במ״מ ו־island קטגוריאלי. הסרת חסרי יעד מותרת; אין להפעיל stratify על כל ערך מסה רציף.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,KFold,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score,mean_absolute_error
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
raw=pd.read_csv(url)
assert len(raw)==344

In [ ]:
data=raw.dropna(subset=[???]).copy()
numeric=[???,???]
categorical=[???]
X=data[numeric+categorical].copy()
y=data[???].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=???,random_state=???)
assert len(data)==342
assert len(X_test)==86
assert set(X_train.index).isdisjoint(set(X_test.index))

## שלב 2 — Pipeline בטוח
השלמה ותקנון למספריים, השלמה וקידוד OneHot ל־island. ב־CV כל שלב נלמד על תת־האימון של אותו קיפול, ולכן אין fit מוקדם על כל הנתונים.

In [ ]:
numeric_pipe=Pipeline([("fill",SimpleImputer(strategy=???)),("scale",StandardScaler())])
categorical_pipe=Pipeline([("fill",SimpleImputer(strategy=???)),
                            ("encode",OneHotEncoder(handle_unknown="ignore",sparse_output=False))])
prepare=ColumnTransformer([("numeric",numeric_pipe,???),("categorical",categorical_pipe,???)])
model=Pipeline([("prepare",prepare),("regression",???())])
assert len(prepare.transformers)==2

## שלב 3 — חמשת קיפולי KFold
כל דוגמת אימון מופיעה פעם אחת בתפקיד אימות. `cross_val_score(...,scoring='r2')` מחזיר ציוני Validation פנימיים, **לא** Test החיצוני.

In [ ]:
cv=KFold(n_splits=???,shuffle=True,random_state=42)
scores=cross_val_score(model,???,???,cv=cv,scoring=???)
assert len(scores)==5
assert np.isfinite(scores).all()
print("Mean CV R²",scores.mean())

## שלב 4 — התאמה מחדש, שאריות ומדדי Test
Fit לכל Train אחרי CV, תחזית אחת ל־Test. שארית היא אמת פחות חיזוי, ו־MAE הוא ממוצע ערכיה המוחלטים. MAE בגרמים, R² חסר יחידות.

In [ ]:
model.fit(???,???)
predictions=model.predict(???)
comparison=pd.DataFrame({"actual_g":y_test,"predicted_g":predictions},index=y_test.index)
comparison["residual_g"]=comparison[???]-comparison[???]
mae_manual=comparison["residual_g"].abs().???()
mae_lib=mean_absolute_error(???,???)
r2=r2_score(???,???)
assert np.isclose(mae_manual,mae_lib)
assert len(comparison)==86
print("MAE g:",mae_lib,"R²:",r2)

## שלב 4ב — קו בסיס הוגן
DummyRegressor חוזה את ממוצע Train; אסור ללמוד את הממוצע של Test למטרת חיזוי. העריכו בדיוק באותו Test.

In [ ]:
baseline=DummyRegressor(strategy=???)
baseline.fit(???,???)
baseline_pred=baseline.predict(???)
baseline_mae=mean_absolute_error(???,???)
baseline_r2=r2_score(???,???)
assert np.isclose(baseline.constant_.ravel()[0],y_train.mean())
print("Baseline MAE g:",baseline_mae,"Baseline R²:",baseline_r2)

## שלב 5 — גרף ומסקנה
ציירו אמת מול חיזוי עם אלכסון y=x. הסבירו למה אין לייחס סיבתיות למקדמי האי, ומה פירוש שגיאה ממוצעת בגרמים.

In [ ]:
plt.figure()
plt.scatter(y_test,predictions,alpha=.65)
limits=[min(y_test.min(),predictions.min()),max(y_test.max(),predictions.max())]
plt.plot(limits,limits,color="black")
plt.xlabel("Actual body mass (g)")
plt.ylabel("Predicted body mass (g)")
plt.title("Regression predictions on held-out Test")
plt.show()
assert mae_lib>=0
assert baseline_mae>=0

## כרטיס הגשה
כתבו: אילו 3 מאפיינים שימשו? מהי יחידת MAE? למה לא stratify על מסה רציפה? מדוע אימות ב־KFold אינו Test? מה ההבדל ב־MAE מול DummyRegressor? האם שינוי seed או מאפיינים אחרי הצצה ל־Test הוא ניסוי תקין?